# M5 Results Summary

Loads all article result JSONs from `$M5_LAB_DIR` and renders:
1. Full model comparison table (WRMSSE by cutoff + geo-mean)
2. Cold-start segmentation — the key finding
3. Per-cutoff LGBM vs Chronos detail
4. Fine-tuning catastrophic forgetting curve

Metric: **WRMSSE** (Weighted Root Mean Squared Scaled Error, 12-level average).  
Cold-start section uses **unweighted median RMSSE per bucket** — revenue weights suppress new-product signal.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

LAB_DIR = os.getenv("M5_LAB_DIR", "/mnt/lab/nmwamsojo")

def load(name):
    path = os.path.join(LAB_DIR, name)
    with open(path) as f:
        return json.load(f)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

## 1  Full model comparison

In [ ]:
naive   = load("naive_baseline_result.json")
zs_gap2 = load("gap2_standalone_result.json")
v20     = load("repro_v20_result.json")
lgbm    = load("lgbm_global_eval_result.json")
blend   = load("blend_lgbm_chronos_result.json")

def geo(vals):
    v = [x for x in vals if x is not None]
    return float(np.prod(v) ** (1 / len(v)))

# Helpers to extract per-cutoff WRMSSE from varied result shapes
def lgbm_by_label(res):
    return {r["label"]: r["wrmsse"] for r in res["results"]}

lgbm_per = lgbm_by_label(lgbm)
blend_per = {r["label"]: r["wrmsse"] for r in blend["results"]}

rows = [
    {
        "Model": "Seasonal Naive (floor)",
        "Autumn": naive["autumn"],
        "Winter": naive["winter"],
        "Spring": naive["spring"],
        "Geo-mean": naive["geo"],
        "Note": "",
    },
    {
        "Model": "Chronos-2 ZS standalone",
        "Autumn": zs_gap2["2015-10-04"]["chronos_standalone"],
        "Winter": zs_gap2["2016-01-03"]["chronos_standalone"],
        "Spring": zs_gap2["2016-04-24"]["chronos_standalone"],
        "Geo-mean": zs_gap2["geo"]["chronos_standalone"],
        "Note": "dept-seg xcl=True, no ensemble",
    },
    {
        "Model": "Chronos-2 tier ensemble",
        "Autumn": zs_gap2["2015-10-04"]["tier"],
        "Winter": zs_gap2["2016-01-03"]["tier"],
        "Spring": zs_gap2["2016-04-24"]["tier"],
        "Geo-mean": zs_gap2["geo"]["tier"],
        "Note": "Low/Med/High ZS tiers",
    },
    {
        "Model": "Chronos-2 + per-dept OLS (v20)",
        "Autumn": v20["expected"]["autumn"],
        "Winter": v20["expected"]["winter"],
        "Spring": v20["expected"]["spring"],
        "Geo-mean": v20["expected"]["geo"],
        "Note": "xcl=True, LOO-3 OLS per dept",
    },
    {
        "Model": "LightGBM global",
        "Autumn": lgbm_per["Autumn"],
        "Winter": lgbm_per["Winter"],
        "Spring": lgbm_per["Spring"],
        "Geo-mean": lgbm["geo"],
        "Note": "37 lag/rolling/price features, Tweedie",
    },
    {
        "Model": "LightGBM × Chronos OLS blend",
        "Autumn": blend_per["Autumn"],
        "Winter": blend_per["Winter"],
        "Spring": blend_per["Spring"],
        "Geo-mean": blend["geo"],
        "Note": "β ≈ 0.85 (LGBM-heavy); no gain",
    },
]

df = pd.DataFrame(rows).set_index("Model")

def fmt(val, col):
    if col == "Note":
        return val
    return f"{val:.4f}"

styled = (
    df.style
    .format({c: "{:.4f}" for c in ["Autumn", "Winter", "Spring", "Geo-mean"]})
    .highlight_min(subset=["Geo-mean"], color="#d4edda")
    .highlight_max(subset=["Geo-mean"], color="#f8d7da")
    .set_caption("WRMSSE (lower = better). Green = best, red = worst.")
)
styled

## 2  Cold-start natural segmentation — the key finding

Metric: **unweighted per-series median RMSSE** (not WRMSSE).  
History bucket = days since first non-zero sale before the test cutoff.  
Chronos signal: pure ZS dept-segmented xcl=True — no fine-tuning, no ensemble.  
This is what a practitioner deploys at day-0 of a new SKU.

In [ ]:
cs = load("coldstart_natural_seg_result.json")
agg = cs["aggregated"]

buckets = list(agg.keys())
cs_rows = []
for b in buckets:
    d = agg[b]
    winner = "Chronos-2 ✓" if d["chronos_geo_median"] < d["lgbm_geo_median"] else "LightGBM ✓"
    cs_rows.append({
        "History bucket": b,
        "n (Autumn)": d["n_autumn"],
        "LightGBM": d["lgbm_geo_median"],
        "Chronos-2 ZS": d["chronos_geo_median"],
        "Chronos wins %": d["chronos_wins_pct"],
        "Winner": winner,
    })

cs_df = pd.DataFrame(cs_rows).set_index("History bucket")

def highlight_winner(row):
    styles = [""] * len(row)
    lgbm_idx  = list(row.index).index("LightGBM")
    chron_idx = list(row.index).index("Chronos-2 ZS")
    if row["LightGBM"] < row["Chronos-2 ZS"]:
        styles[lgbm_idx] = "background-color: #d4edda; font-weight: bold"
    else:
        styles[chron_idx] = "background-color: #d4edda; font-weight: bold"
    return styles

(
    cs_df.style
    .format({"LightGBM": "{:.3f}", "Chronos-2 ZS": "{:.3f}", "Chronos wins %": "{:.1f}%"})
    .apply(highlight_winner, axis=1)
    .set_caption(
        "Geo-mean of median RMSSE per bucket (across 3 cutoffs). "
        "Green = lower RMSSE (better). "
        "Crossover at 28d matches lag_28 availability in LightGBM features."
    )
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

x   = np.arange(len(buckets))
w   = 0.35
lgbm_vals   = [agg[b]["lgbm_geo_median"]    for b in buckets]
chronos_vals = [agg[b]["chronos_geo_median"] for b in buckets]
win_pct      = [agg[b]["chronos_wins_pct"]   for b in buckets]

# Left: RMSSE by bucket
ax = axes[0]
bars_l = ax.bar(x - w/2, lgbm_vals,    w, label="LightGBM",   color="#4C72B0", alpha=0.85)
bars_c = ax.bar(x + w/2, chronos_vals, w, label="Chronos-2",  color="#DD8452", alpha=0.85)
ax.axvline(0.5, color="red", linewidth=1.5, linestyle="--", label="28d crossover")
ax.set_xticks(x)
ax.set_xticklabels(buckets, rotation=20, ha="right")
ax.set_ylabel("Median RMSSE (geo-mean across cutoffs)")
ax.set_title("RMSSE by active history length")
ax.legend()
ax.set_ylim(0, max(max(lgbm_vals), max(chronos_vals)) * 1.15)

# Right: % series where Chronos beats LGBM
ax2 = axes[1]
colors = ["#DD8452" if p > 50 else "#4C72B0" for p in win_pct]
ax2.bar(x, win_pct, color=colors, alpha=0.85)
ax2.axhline(50, color="black", linewidth=1, linestyle=":")
ax2.axvline(0.5, color="red", linewidth=1.5, linestyle="--", label="28d crossover")
ax2.set_xticks(x)
ax2.set_xticklabels(buckets, rotation=20, ha="right")
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.set_ylabel("% series where Chronos-2 wins")
ax2.set_title("Chronos-2 win rate by bucket")
ax2.set_ylim(0, 100)
ax2.legend()

plt.tight_layout()
plt.savefig("../reports/figures/coldstart_segmentation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/coldstart_segmentation.png")

## 3  Per-cutoff breakdown — LGBM vs Chronos v20

In [ ]:
cutoff_labels  = ["Autumn\n(2015-10-04)", "Winter\n(2016-01-03)", "Spring\n(2016-04-24)"]
lgbm_cutoffs   = [lgbm_per["Autumn"],   lgbm_per["Winter"],   lgbm_per["Spring"]]
chronos_cutoffs = [v20["expected"]["autumn"], v20["expected"]["winter"], v20["expected"]["spring"]]
blend_cutoffs  = [blend_per["Autumn"],  blend_per["Winter"],  blend_per["Spring"]]
naive_cutoffs  = [naive["autumn"],      naive["winter"],      naive["spring"]]

fig, ax = plt.subplots(figsize=(8, 4.5))
xi = np.arange(3)
w  = 0.2

ax.bar(xi - 1.5*w, naive_cutoffs,   w, label="Seasonal Naive",       color="#8c8c8c", alpha=0.7)
ax.bar(xi - 0.5*w, chronos_cutoffs, w, label="Chronos-2 v20 OLS",    color="#DD8452", alpha=0.85)
ax.bar(xi + 0.5*w, lgbm_cutoffs,    w, label="LightGBM global",       color="#4C72B0", alpha=0.85)
ax.bar(xi + 1.5*w, blend_cutoffs,   w, label="LGBM × Chronos blend",  color="#55A868", alpha=0.7)

ax.set_xticks(xi)
ax.set_xticklabels(cutoff_labels)
ax.set_ylabel("WRMSSE (lower = better)")
ax.set_title("WRMSSE per evaluation cutoff")
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(0, max(naive_cutoffs) * 1.1)

plt.tight_layout()
plt.savefig("../reports/figures/wrmsse_per_cutoff.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/wrmsse_per_cutoff.png")

## 4  Fine-tuning: catastrophic forgetting curve

Why we rely on zero-shot Chronos rather than fine-tuning: performance degrades monotonically after ~5 gradient steps.

In [ ]:
try:
    sc = load("steps_curve_results.json")
    steps  = [r["steps"]   for r in sc["results"]]
    scores = [r["wrmsse"]  for r in sc["results"]]
    zs_val = sc.get("zs_baseline")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(steps, scores, marker="o", color="#DD8452", linewidth=2, label="Fine-tuned WRMSSE")
    if zs_val:
        ax.axhline(zs_val, color="#4C72B0", linestyle="--", linewidth=1.5, label=f"ZS baseline ({zs_val:.4f})")
    ax.set_xlabel("Fine-tuning gradient steps")
    ax.set_ylabel("WRMSSE")
    ax.set_title("Chronos-2 fine-tuning: catastrophic forgetting")
    ax.legend()
    plt.tight_layout()
    plt.savefig("../reports/figures/ft_steps_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → reports/figures/ft_steps_curve.png")
except FileNotFoundError:
    print("steps_curve_results.json not found in LAB_DIR — run experiment 11 first.")

## 5  Headline numbers (copy-paste ready)

In [ ]:
print("── Key results ──────────────────────────────────────────")
print(f"  Seasonal Naive geo-mean WRMSSE : {naive['geo']:.4f}  (floor)")
print(f"  Chronos-2 standalone ZS        : {zs_gap2['geo']['chronos_standalone']:.4f}")
print(f"  Chronos-2 v20 OLS (best)       : {v20['expected']['geo']:.4f}")
print(f"  LightGBM global (best)         : {lgbm['geo']:.4f}")
print(f"  LGBM × Chronos blend           : {blend['geo']:.4f}  (no gain vs LGBM)")
print()
print("── Cold-start (<28d) ────────────────────────────────────")
cs_lt28 = agg["<28d"]
print(f"  LightGBM  median RMSSE : {cs_lt28['lgbm_geo_median']:.4f}")
print(f"  Chronos-2 median RMSSE : {cs_lt28['chronos_geo_median']:.4f}")
print(f"  Chronos-2 wins         : {cs_lt28['chronos_wins_pct']:.1f}% of series")
print(f"  n (<28d, Autumn)       : {cs_lt28['n_autumn']}")
print()
print("── Deployment rule ──────────────────────────────────────")
print("  Use Chronos-2 ZS for the first 28 days after SKU launch.")
print("  Switch to LightGBM once lag_28 becomes available.")
print("  Crossover is mechanistically justified, not just empirical.")